# 第 21 章 · baseline 与 reward-to-go：给策略梯度降噪

本章回答：REINFORCE 为什么学得这么慢？减掉一个"基准线"为什么不会把梯度弄错，却能大幅降噪？为什么一个动作只该为它之后的奖励负责？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/21-baseline.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np
import torch

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 不减 baseline，2 万个样本 | `grad_samples(0.0)` | 平均 0.286，方差 27.85 |
| 减 b = 10.5 | `grad_samples(10.5)` | 平均 0.254，方差 0.25 |
| reward-to-go | `reward_to_go([1, 0, 2, 0])` | [3, 2, 2, 0] |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `softmax`**（来自 `ch20_policy_gradient.py`）：下面要用到的小工具。

In [ ]:
def softmax(theta):
    e = np.exp(theta - theta.max())
    return e / e.sum()

**函数 `grad_log_softmax`**（来自 `ch20_policy_gradient.py`）：∇_θ log π(a) = one_hot(a) − π（softmax 策略的现成结论，下面用 autograd 核对）。

In [ ]:
def grad_log_softmax(theta, a):
    """∇_θ log π(a) = one_hot(a) − π（softmax 策略的现成结论，下面用 autograd 核对）。"""
    g = -softmax(theta)
    g[a] += 1
    return g

**函数 `grad_samples`**（来自 `ch21_baseline.py`）：两台机器奖励都在 10 左右。返回 n 个单样本梯度估计（只看 θ[1] 那一维）。

In [ ]:
def grad_samples(baseline, n=20000, seed=0, means=(10.0, 11.0)):
    """两台机器奖励都在 10 左右。返回 n 个单样本梯度估计（只看 θ[1] 那一维）。"""
    rng, theta, out = np.random.default_rng(seed), np.zeros(2), []
    pi = softmax(theta)
    for _ in range(n):
        a = rng.choice(2, p=pi)
        G = rng.normal(means[a], 1.0)
        out.append((G - baseline) * grad_log_softmax(theta, a)[1])
    return np.array(out)

**函数 `reward_to_go`**（来自 `ch21_baseline.py`）：每一步只用它之后的奖励：G_t = Σ_{t' ≥ t} γ^{t'−t} r_{t'}。

In [ ]:
def reward_to_go(rewards, gamma=1.0):
    """每一步只用它之后的奖励：G_t = Σ_{t' ≥ t} γ^{t'−t} r_{t'}。"""
    out, G = [], 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    return out[::-1]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
for b in [0.0, 10.5]:
        g = grad_samples(b)
        print(f"baseline = {b}：梯度平均 {g.mean():.3f}，方差 {g.var():.2f}")

In [ ]:
r = [1, 0, 2, 0]

In [ ]:
print("奖励", r, "：整条回报都是", sum(r), "；reward-to-go", reward_to_go(r))

## 3. 代码里才看得到的

`reward_to_go` 和第 3 章的 `returns_backward`、第 14 章 MC 的倒着走是同一段代码。CS 285 HW2 的 `_discounted_reward_to_go` 也长这样。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换 baseline：`grad_samples(b)`，b = 5 → 方差 7.83；b = 10.5 → 0.25；b = 20 → 22.78。平均值都在 0.25 左右（不偏），方差在 b 接近平均奖励时最小。
2. `reward_to_go([1, 0, 2, 0], 0.9)` → [2.62, 1.8, 2.0, 0.0]：加上折扣，就是第 3 章倒着推的那个循环。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 奖励全是正的，每个动作都推高，信号淹没在噪声里；
> 减个 baseline：只看比平常好多少，好的推高差的压低；
> 概率加起来是一，导数是零，所以减 b 不改方向只降噪；
> 动作只管身后事：reward-to-go 就是 $G_t$；b 随状态变就是价值网络。